# Orbit Wars Transformer — Hellburner heuristics fixed labels v2

Bản này giữ pipeline gốc của `orbit-transformer.ipynb`:

1. Parse replay thành dataset `(X, MASK, Y)`.
2. Train `CandidateTransformer` để rank target.
3. Save `winner_each_replay_transformer_best.pt` và `feature_stats.npz`.

Điểm thay đổi chính:

- Thêm các feature lấy cảm hứng trực tiếp từ `82_hellburner_upgraded.py`:
  - intercept planet quay;
  - kiểm tra đường bay cắt Sun;
  - first-planet-hit;
  - required ships theo ETA + production;
  - source garrison/surplus safety;
  - ROI score;
  - defense urgency;
  - pressure/counter-pressure quanh target.
- Mask candidate nguy hiểm ở lúc tạo dataset, ví dụ đường bắn xuyên Sun hoặc bắn nhầm planet khác.
- Sửa model để `feat_dim` lấy động từ shard, không hard-code 60 nữa.

Sau khi chạy lại cell parse replay, bạn sẽ có shard mới với `FEATURE_DIM` lớn hơn bản cũ, vì vậy **phải train lại model**.


## Bản sửa lỗi label/noise v2

Bản này đã sửa các lỗi quan trọng làm model học sai:

- `infer_target_by_angle_lead()` không còn dùng tốc độ cố định `6.0`; ETA để suy label giờ dùng `label_fleet_speed(ships)` giống công thức tốc độ fleet theo số quân.
- 5 feature trước đây bị hard-code `0.0` đã được tính xấp xỉ thật: `threat_after`, `support_after`, `pre_volatility`, `orbital_trend`, `convergence_threat`.
- `source_safe` không còn là hard-mask bắt buộc trong `hb_candidate_valid`; nó được giữ làm feature/penalty để tránh loại quá nhiều nước đi tốt ở early game.
- Garrison safety được nới theo phase game, đặc biệt early game không giữ cứng 8 quân.
- Training thêm `label_smoothing` và cosine LR scheduler để ổn định hơn.

**Lưu ý:** Vì label và feature đã đổi, cần parse replay lại từ đầu rồi train lại model.


In [1]:
import os, json, math, glob, argparse
import numpy as np
from collections import defaultdict, deque
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# =============================================================================
# CONSTANTS
# =============================================================================
BOARD        = 100.0
CENTER       = 50.0
TOTAL_STEPS  = 500.0
MAX_ETA      = 25.0
EPS          = 1e-6
K_SHIPS      = 40.0
SHIP_SPEED   = 6.0
LAMBDA_ECEP  = 0.3
LAMBDA_POT   = 0.15
MAX_LABEL_ANGLE_ERR = 0.05

# Định nghĩa tên các đặc trưng (features) được sử dụng trong mô hình
FEATURE_NAMES = [
    'game_progress', # tiến trình của trò chơi, có thể là phần trăm hoàn thành hoặc số cấp độ đã đạt được
    'players_alive', # số lượng người chơi còn sống
    'my_planet_share', # tỷ lệ phần trăm hành tinh mà người chơi sở hữu so với tổng số hành tinh
    'my_gdp_share', # tỷ lệ phần trăm GDP (Gross Domestic Product) của người chơi so với tổng GDP của tất cả người chơi
    'my_ship_share', # tỷ lệ phần trăm số lượng tàu của người chơi so với tổng số tàu của tất cả người chơi
    'momentum', # động lượng, có thể là sự thay đổi về số lượng tàu hoặc hành tinh trong một khoảng thời gian nhất định
    'my_fleet_ratio', # tỷ lệ giữa số lượng tàu của người chơi và số lượng tàu của đối thủ
    'leader_gap', # khoảng cách giữa người chơi và người dẫn đầu, có thể được tính bằng cách so sánh số lượng tàu, hành tinh hoặc GDP của người chơi với người dẫn đầu
    'am_i_targeted', # một đặc trưng nhị phân cho biết liệu người chơi có đang bị đối thủ nhắm đến hay không, có thể dựa trên số lượng tàu của đối thủ hướng về phía người chơi hoặc các hành động tấn công gần đây
    'fastest_grower_gap', # khoảng cách giữa người chơi và người phát triển nhanh nhất, có thể được tính bằng cách so sánh tốc độ tăng trưởng của người chơi với người phát triển nhanh nhất trong trò chơi
    'aggression_trend', # xu hướng tấn công, có thể được tính bằng cách phân tích các hành động tấn công gần đây của người chơi và đối thủ để xác định xem người chơi có đang trở nên hung hăng hơn hay không
    'enemy_rhythm', # nhịp độ của đối thủ, có thể được tính bằng cách phân tích tần suất và cường độ của các hành động tấn công của đối thủ để xác định xem họ có đang chơi một cách nhanh chóng và hung hăng hay không
    'src_ships', # số lượng tàu của người chơi tại thời điểm hiện tại
    'src_production', # sản lượng của người chơi tại thời điểm hiện tại
    'src_safety', # mức độ an toàn của người chơi, có thể được tính bằng cách phân tích số lượng tàu của đối thủ hướng về phía người chơi hoặc các hành động tấn công gần đây để xác định xem người chơi có đang ở trong tình trạng nguy hiểm hay không
    'target_diplomacy', # một đặc trưng nhị phân cho biết liệu người chơi có đang trong một liên minh ngoại giao với đối thủ hay không, có thể dựa trên các hành động ngoại giao gần đây hoặc trạng thái liên minh hiện tại của người chơi
    'target_production', # sản lượng của đối thủ tại thời điểm hiện tại
    'is_comet', # một đặc trưng nhị phân cho biết liệu người chơi có đang bị tấn công bởi một hành tinh sao chổi hay không, có thể dựa trên số lượng tàu của đối thủ hướng về phía người chơi hoặc các hành động tấn công gần đây liên quan đến hành tinh sao chổi
    'owner_power', # sức mạnh của người chơi, có thể được tính bằng cách kết hợp các đặc trưng khác như số lượng tàu, hành tinh và GDP để tạo ra một chỉ số tổng hợp về sức mạnh của người chơi
    'is_weakest_target', # một đặc trưng nhị phân cho biết liệu người chơi có đang là mục tiêu yếu nhất của đối thủ hay không, có thể dựa trên việc so sánh sức mạnh của người chơi với các đối thủ khác để xác định xem người chơi có đang bị nhắm đến như một mục tiêu dễ dàng hay không
    'projected_owner', # một đặc trưng dự đoán về chủ sở hữu của hành tinh mục tiêu trong tương lai, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem ai sẽ kiểm soát hành tinh mục tiêu trong tương lai
    'projected_resistance', # một đặc trưng dự đoán về khả năng kháng cự của hành tinh mục tiêu trong tương lai, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem hành tinh mục tiêu sẽ có khả năng kháng cự như thế nào trong tương lai
    'threat_after', # một đặc trưng dự đoán về mức độ đe dọa mà người chơi sẽ phải đối mặt sau khi thực hiện một hành động cụ thể, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi sẽ phải đối mặt với những mối đe dọa nào sau khi thực hiện một hành động cụ thể
    'support_after', # một đặc trưng dự đoán về mức độ hỗ trợ mà người chơi sẽ nhận được sau khi thực hiện một hành động cụ thể, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi sẽ nhận được những hỗ trợ nào sau khi thực hiện một hành động cụ thể
    'pre_volatility', # một đặc trưng dự đoán về mức độ biến động của trò chơi trước khi thực hiện một hành động cụ thể, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem trò chơi sẽ trở nên biến động như thế nào trước khi thực hiện một hành động cụ thể
    'threat_potential', # một đặc trưng dự đoán về tiềm năng đe dọa mà người chơi có thể đối mặt trong tương lai, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi có thể phải đối mặt với những mối đe dọa nào trong tương lai
    'support_potential', # một đặc trưng dự đoán về tiềm năng hỗ trợ mà người chơi có thể nhận được trong tương lai, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi có thể nhận được những hỗ trợ nào trong tương lai
    'eta', # thời gian ước tính để hoàn thành một hành động cụ thể, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi sẽ mất bao lâu để hoàn thành một hành động cụ thể
    'commitment_ratio', # tỷ lệ cam kết của người chơi, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi có đang cam kết với một chiến lược cụ thể hay không
    'required_ratio', # tỷ lệ yêu cầu, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi sẽ cần phải đáp ứng những yêu cầu nào để thực hiện một hành động cụ thể
    'margin', # một đặc trưng dự đoán về khoảng cách giữa người chơi và đối thủ sau khi thực hiện một hành động cụ thể, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi sẽ có khoảng cách như thế nào so với đối thủ sau khi thực hiện một hành động cụ thể
    'can_win', # một đặc trưng nhị phân cho biết liệu người chơi có thể giành chiến thắng ngay sau khi thực hiện một hành động cụ thể hay không, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của người chơi và đối thủ để dự đoán xem người chơi có thể giành chiến thắng ngay sau khi thực hiện một hành động cụ thể hay không
    'garrison_strength', # sức mạnh của lực lượng đồn trú, có thể được tính bằng cách phân tích số lượng tàu và hành tinh mà người chơi sở hữu để tạo ra một chỉ số tổng hợp về sức mạnh của lực lượng đồn trú của người chơi
    'defense_sustainability', # khả năng duy trì phòng thủ, có thể được tính bằng cách phân tích số lượng tàu và hành tinh mà người chơi sở hữu để tạo ra một chỉ số tổng hợp về khả năng duy trì phòng thủ của người chơi
    'target_roi', # lợi tức đầu tư của mục tiêu, có thể được tính bằng cách phân tích số lượng tàu và hành tinh mà người chơi sở hữu để tạo ra một chỉ số tổng hợp về lợi tức đầu tư của mục tiêu
    'front_gradient', # một đặc trưng đo lường độ dốc của chiến tuyến, có thể được tính bằng cách phân tích vị trí và hướng di chuyển của các tàu
    'avg_enemy_distance', # khoảng cách trung bình đến các đối thủ, có thể được tính bằng cách phân tích vị trí của các tàu đối thủ
    'angular_spread', # sự phân bố góc của các tàu đối thủ, có thể được tính bằng cách phân tích góc giữa các tàu đối thủ
    'orbital_trend', # xu hướng quỹ đạo của các tàu, có thể được tính bằng cách phân tích chuyển động của các tàu theo thời gian
    'convergence_threat', # mức độ đe dọa từ sự hội tụ của các tàu đối thủ, có thể được tính bằng cách phân tích vị trí và hướng di chuyển của các tàu đối thủ
    'approach_rate', # tốc độ tiếp cận của các tàu đối thủ, có thể được tính bằng cách phân tích chuyển động của các tàu theo thời gian
    'enemy_commitment', # mức độ cam kết của các tàu đối thủ, có thể được tính bằng cách phân tích các hành động gần đây và xu hướng của đối thủ
    'weakest_colony', # hành tinh yếu nhất của người chơi, có thể được xác định bằng cách phân tích vị trí và sức mạnh của các hành tinh
    'local_superiority', # sự ưu thế địa phương, có thể được tính bằng cách phân tích vị trí và sức mạnh của các tàu trong khu vực cụ thể
    'economic_impact', # tác động kinh tế, có thể được tính bằng cách phân tích số lượng tàu và hành tinh mà người chơi sở hữu để tạo ra một chỉ số tổng hợp về tác động kinh tế của người chơi
    'enemy_exposed' # một đặc trưng nhị phân cho biết liệu đối thủ có đang bị phơi bày hay không, có thể được tính bằng cách phân tích vị trí và hướng di chuyển của các tàu đối thủ để xác định xem họ có đang ở trong tình trạng dễ bị tấn công hay không
]


# =============================================================================
# MATH UTILS
# =============================================================================
def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, float(x)))

def ship_sigmoid(x):
    x = max(0.0, float(x))
    return x / (x + K_SHIPS)

def dist_xy(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)

def angle_diff(a, b):
    """Khoảng cách góc nhỏ nhất (absolute), kết quả trong [0, pi]."""
    d = (a - b + math.pi) % (2 * math.pi) - math.pi
    return abs(d)

In [2]:
# =============================================================================
# BƯỚC 1 – ORBITAL MECHANICS
# =============================================================================
def is_orbiting_planet(p):
    """
    Planet quay quanh tâm nếu khoảng cách tâm + radius < 50.
    Planet format: [id, owner, x, y, radius, ships, production]
    """
    _, _, x, y, radius, *_ = p
    return dist_xy(x, y, CENTER, CENTER) + radius < 50.0


def predict_rotated_xy(p, delta_step, angular_velocity):
    """
    Tính vị trí (x, y) của planet sau delta_step turns.
    Planet tĩnh (không orbit) trả về vị trí hiện tại.
    """
    _, _, x, y, radius, *_ = p
    if not is_orbiting_planet(p):
        return x, y
    dx = x - CENTER
    dy = y - CENTER
    r  = math.hypot(dx, dy)
    theta = math.atan2(dy, dx) + angular_velocity * delta_step
    return CENTER + r * math.cos(theta), CENTER + r * math.sin(theta)

# =============================================================================
# BƯỚC 2 – TÌM TARGET TỪ ANGLE
# =============================================================================
def label_fleet_speed(ships):
    """Tốc độ fleet dùng khi suy label từ replay.

    Bug cũ: estimate_eta dùng SHIP_SPEED=6 cố định, làm label target sai
    với fleet nhỏ vì game cho fleet nhỏ bay chậm hơn. Công thức này khớp
    với hb_fleet_speed/Hellburner-style speed.
    """
    ships = max(1.0, float(ships))
    return min(
        SHIP_SPEED,
        1.0 + (SHIP_SPEED - 1.0) * (math.log(ships) / math.log(1000.0)) ** 1.5,
    )


def estimate_eta(src, tgt, ships):
    d = dist_xy(src[2], src[3], tgt[2], tgt[3])
    speed = max(EPS, label_fleet_speed(ships))
    return max(1, int(math.ceil(d / speed))), d


def infer_target_by_angle_lead(src, planets, action_angle, ships, angular_velocity,
                                max_error=0.25):
    """
    Suy ra target_id từ action angle.

    Cải tiến so với pipeline cũ:
    - Test 3 vị trí ứng viên cho mỗi planet (vị trí hiện tại, sau eta turns,
      sau eta//2 turns) để bắt được các trường hợp bắn lead vào planet đang quay.
    - ETA dùng tốc độ fleet theo ships, không còn dùng tốc độ cố định 6.0.
    - Threshold chặt hơn (0.25 rad vs 1.5 rad cũ), giảm false positive.

    Trả về (target_id, best_err) hoặc (None, err) nếu không tìm thấy.
    """
    best_id  = None
    best_err = 1e9
    sx, sy   = src[2], src[3]

    for tgt in planets:
        if int(tgt[0]) == int(src[0]):
            continue

        eta, _ = estimate_eta(src, tgt, ships)

        # 3 vị trí candidate: hiện tại, sau eta turns, sau eta//2 turns
        candidate_positions = [
            (tgt[2], tgt[3]),
            predict_rotated_xy(tgt, eta,              angular_velocity),
            predict_rotated_xy(tgt, max(1, eta // 2), angular_velocity),
        ]

        for tx, ty in candidate_positions:
            expected = math.atan2(ty - sy, tx - sx)
            err      = angle_diff(action_angle, expected)
            if err < best_err:
                best_err = err
                best_id  = int(tgt[0])

    if best_err > max_error:
        return None, best_err
    return best_id, best_err


# =============================================================================
# BƯỚC 3 – CANDIDATE GENERATION
# =============================================================================
def build_candidates(src, planets, K=24):
    """
    Candidate generator stratified (từ GPT, tốt hơn nearest-only).

    Ưu tiên:
    - Enemy:   K//3 gần nhất (muốn tấn công)
    - Neutral: K//2 có tỷ lệ ships/prod thấp nhất (dễ chiếm, sinh lời cao)
    - Friendly: K//4 gần nhất (có thể reinforce)
    - Fill bằng nearest chưa được chọn

    Quan trọng: đảm bảo true_target luôn nằm trong tập candidates.
    """
    src_id = int(src[0])
    player = int(src[1])
    sx, sy = src[2], src[3]

    rows = []
    for p in planets:
        if int(p[0]) == src_id:
            continue
        d     = dist_xy(sx, sy, p[2], p[3])
        owner = int(p[1])
        rows.append((p, d, owner))

    enemy   = [(p, d) for p, d, o in rows if o not in (-1, player)]
    neutral = [(p, d) for p, d, o in rows if o == -1]
    friendly= [(p, d) for p, d, o in rows if o == player]

    enemy.sort(key=lambda x: x[1])
    # Neutral ưu tiên theo ships/prod (thấp = dễ chiếm, sinh lời cao)
    neutral.sort(key=lambda x: (x[0][5] / (x[0][6] + EPS), x[1]))
    friendly.sort(key=lambda x: x[1])

    selected    = []
    selected_ids= set()

    for bucket in [enemy[:K//3], neutral[:K//2], friendly[:K//4]]:
        for p, d in bucket:
            pid = int(p[0])
            if pid not in selected_ids:
                selected.append((p, d))
                selected_ids.add(pid)

    # Fill bằng nearest chưa có
    all_nearest = sorted([(p, d) for p, d, _ in rows], key=lambda x: x[1])
    for p, d in all_nearest:
        if len(selected) >= K:
            break
        pid = int(p[0])
        if pid not in selected_ids:
            selected.append((p, d))
            selected_ids.add(pid)

    return selected[:K]

In [3]:

# =============================================================================
# BƯỚC 3.5 – HELLBURNER-STYLE HEURISTIC FEATURES / GUARDRAILS
# =============================================================================
# Các hàm này được rút gọn từ ý tưởng trong 82_hellburner_upgraded.py:
# - fleet_speed theo số quân
# - intercept planet quay
# - first_planet_hit để tránh bắn nhầm planet
# - point_to_segment_distance để tránh đường bay xuyên Sun
# - ROI / source safety / required ships
#
# Lưu ý:
# - Đây không phải nhúng nguyên agent Hellburner.
# - Transformer vẫn học target ranking.
# - Hellburner heuristics chỉ bổ sung feature + mask candidate nguy hiểm.

SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
SHIP_SPEED_MAX = 6.0
GAME_LENGTH = 500.0

HELLBURNER_FEATURE_NAMES = [
    "hb_intercept_eta_norm",        # ETA khi bắn lead vào target quay
    "hb_intercept_angle_sin",       # sin(angle intercept)
    "hb_intercept_angle_cos",       # cos(angle intercept)
    "hb_crosses_sun",               # đường từ src đến điểm intercept cắt Sun
    "hb_first_hit_is_target",       # ray bắn đầu tiên có chạm đúng target không
    "hb_required_ships_norm",       # số quân heuristic cần gửi
    "hb_required_ratio",            # required / src_ships
    "hb_surplus_after_send_norm",   # src còn dư bao nhiêu sau khi gửi required
    "hb_source_safe_after_send",    # source vẫn giữ được garrison tối thiểu
    "hb_roi_score",                 # production * payoff_turns / cost
    "hb_payoff_turns_norm",         # số turn còn sinh lời sau khi đến nơi
    "hb_defense_urgency",           # target của mình đang bị đe dọa
    "hb_target_enemy_pressure",     # áp lực enemy quanh target
    "hb_target_ally_support",       # support của mình quanh target
    "hb_local_pressure_ratio",      # support / pressure
    "hb_target_is_orbiting",        # target có quay quanh Sun không
    "hb_source_is_frontline",       # source gần enemy / frontier
    "hb_counter_risk",              # enemy gần target có thể counter không
]


def hb_fleet_speed(ships):
    """Công thức tốc độ fleet giống Hellburner."""
    ships = max(1.0, float(ships))
    return min(
        SHIP_SPEED_MAX,
        1.0 + (SHIP_SPEED_MAX - 1.0) * (math.log(ships) / math.log(1000.0)) ** 1.5,
    )


def hb_point_to_segment_distance(px, py, ax, ay, bx, by):
    """Khoảng cách từ điểm P đến đoạn AB."""
    abx = bx - ax
    aby = by - ay
    apx = px - ax
    apy = py - ay
    denom = abx * abx + aby * aby
    if denom <= EPS:
        return math.hypot(px - ax, py - ay)
    t = max(0.0, min(1.0, (apx * abx + apy * aby) / denom))
    cx = ax + t * abx
    cy = ay + t * aby
    return math.hypot(px - cx, py - cy)


def hb_segment_crosses_sun(ax, ay, bx, by, margin=0.0):
    return hb_point_to_segment_distance(CENTER, CENTER, ax, ay, bx, by) <= (SUN_RADIUS + margin)


def hb_is_orbiting(p):
    return dist_xy(float(p[2]), float(p[3]), CENTER, CENTER) + float(p[4]) < ROTATION_RADIUS_LIMIT


def hb_intercept_planet(src, tgt, ships, angular_velocity, scene_step, tol=1e-6, max_iters=30):
    """
    Dạng rút gọn từ Hellburner.intercept_planet().
    Trả về: angle, intercept_x, intercept_y, travel_turns.
    """
    sx, sy = float(src[2]), float(src[3])
    speed = hb_fleet_speed(ships)

    if not hb_is_orbiting(tgt):
        tx, ty = float(tgt[2]), float(tgt[3])
        travel = dist_xy(sx, sy, tx, ty) / speed
    else:
        # Không có object orbital_info như Hellburner, nên lấy góc hiện tại làm mốc.
        r = dist_xy(float(tgt[2]), float(tgt[3]), CENTER, CENTER)
        cur_angle = math.atan2(float(tgt[3]) - CENTER, float(tgt[2]) - CENTER)

        # Ước lượng lặp ETA -> vị trí target tương lai.
        travel = dist_xy(sx, sy, float(tgt[2]), float(tgt[3])) / speed
        for _ in range(max_iters):
            a = cur_angle + angular_velocity * max(0.0, travel - 0.5)
            tx = CENTER + r * math.cos(a)
            ty = CENTER + r * math.sin(a)
            new_travel = dist_xy(sx, sy, tx, ty) / speed
            # damping giống tinh thần Hellburner để tránh dao động.
            new_travel = 0.5 * (travel + new_travel)
            if abs(new_travel - travel) < tol:
                travel = new_travel
                break
            travel = new_travel
        else:
            return 0.0, float(tgt[2]), float(tgt[3]), math.inf

        a = cur_angle + angular_velocity * max(0.0, travel - 0.5)
        tx = CENTER + r * math.cos(a)
        ty = CENTER + r * math.sin(a)

    angle = math.atan2(ty - sy, tx - sx)
    return angle, tx, ty, travel


def hb_first_planet_hit(src, angle, ships, planets, angular_velocity, scene_step):
    """
    Dạng rút gọn từ Hellburner.first_planet_hit().
    Tìm planet đầu tiên mà ray/fleet sẽ chạm theo angle.
    Nếu đường đến hit đầu tiên cắt Sun -> trả về None.
    """
    sx, sy = float(src[2]), float(src[3])
    source_id = int(src[0])

    best = None
    best_t = float("inf")

    for p in planets:
        if int(p[0]) == source_id:
            continue

        needed_angle, px, py, travel = hb_intercept_planet(
            src, p, ships, angular_velocity, scene_step
        )

        if not math.isfinite(travel):
            continue

        d = dist_xy(sx, sy, px, py)
        radius = float(p[4])
        half_cone = math.pi if d < radius else math.asin(min(1.0, radius / max(d, EPS)))
        delta = angle_diff(angle, needed_angle)

        if delta <= half_cone and travel < best_t:
            best_t = travel
            best = p

    if best is None:
        return None

    ex = sx + best_t * hb_fleet_speed(ships) * math.cos(angle)
    ey = sy + best_t * hb_fleet_speed(ships) * math.sin(angle)

    if hb_segment_crosses_sun(sx, sy, ex, ey):
        return None

    return best


def hb_required_ships(src, tgt, travel):
    """
    Heuristic ship sizing:
    - friendly: reinforce nhỏ, không phải attack;
    - neutral: tgt.ships + 1 + buffer;
    - enemy: tgt.ships + production trong ETA + buffer.
    """
    player = int(src[1])
    tgt_owner = int(tgt[1])
    tgt_ships = float(tgt[5])
    tgt_prod = float(tgt[6])

    eta_turns = max(1, int(math.ceil(travel))) if math.isfinite(travel) else 99

    if tgt_owner == player:
        return max(0.0, min(float(src[5]) - 3.0, 12.0))

    if tgt_owner == -1:
        return tgt_ships + 1.0 + max(1.0, 0.15 * tgt_ships)

    # Enemy planet sẽ sinh thêm quân trước khi fleet tới.
    return tgt_ships + tgt_prod * eta_turns + 2.0 + max(1.0, 0.10 * tgt_ships)


def hb_local_pressure(tgt, planets, player):
    """
    Ước lượng pressure/support quanh target dựa trên ships và khoảng cách.
    """
    tx, ty = float(tgt[2]), float(tgt[3])
    enemy_pressure = 0.0
    ally_support = 0.0

    for p in planets:
        if int(p[0]) == int(tgt[0]):
            continue
        d = max(1.0, dist_xy(float(p[2]), float(p[3]), tx, ty))
        influence = ship_sigmoid(float(p[5])) * math.exp(-d / 35.0)

        if int(p[1]) == player:
            ally_support += influence
        elif int(p[1]) != -1:
            enemy_pressure += influence

    return enemy_pressure, ally_support


def hb_source_is_frontline(src, planets, player):
    sx, sy = float(src[2]), float(src[3])
    nearest_enemy = min(
        (
            dist_xy(sx, sy, float(p[2]), float(p[3]))
            for p in planets
            if int(p[1]) not in (-1, player)
        ),
        default=BOARD,
    )
    return 1.0 if nearest_enemy <= 38.0 else 0.0


def hb_candidate_metrics(src, tgt, g):
    """
    Tính toàn bộ Hellburner-style metrics cho một candidate.
    """
    planets = g.get("planets", [])
    player = int(src[1])
    scene_step = int(g.get("step_idx", 0))
    angular_velocity = float(g.get("angular_velocity", 0.0))

    src_ships = max(1.0, float(src[5]))

    # Tính theo required sơ bộ rồi intercept lại bằng required.
    coarse_angle, coarse_ix, coarse_iy, coarse_travel = hb_intercept_planet(
        src, tgt, max(1.0, src_ships), angular_velocity, scene_step
    )
    if not math.isfinite(coarse_travel):
        return {
            "valid": False,
            "features": [0.0] * len(HELLBURNER_FEATURE_NAMES),
        }

    required = hb_required_ships(src, tgt, coarse_travel)
    required = max(1.0, min(required, src_ships))

    angle, ix, iy, travel = hb_intercept_planet(
        src, tgt, required, angular_velocity, scene_step
    )

    if not math.isfinite(travel):
        return {
            "valid": False,
            "features": [0.0] * len(HELLBURNER_FEATURE_NAMES),
        }

    crosses_sun = 1.0 if hb_segment_crosses_sun(float(src[2]), float(src[3]), ix, iy) else 0.0

    first_hit = hb_first_planet_hit(
        src, angle, required, planets, angular_velocity, scene_step
    )
    first_hit_is_target = 1.0 if (first_hit is not None and int(first_hit[0]) == int(tgt[0])) else 0.0

    # Source safety là FEATURE/PENALTY, không nên hard-mask tuyệt đối.
    # Bug cũ: giữ cứng 8 quân làm early game gần như không dám chiếm neutral.
    if scene_step < 75:
        garrison_keep = 2.0
    elif scene_step < 250:
        garrison_keep = 5.0
    elif scene_step < 440:
        garrison_keep = 8.0
    else:
        garrison_keep = 1.0

    source_frontline = hb_source_is_frontline(src, planets, player)
    if source_frontline > 0.5 and scene_step >= 75:
        garrison_keep += 2.0

    surplus_after = src_ships - required
    source_safe = 1.0 if surplus_after >= garrison_keep else 0.0

    turns_remaining = max(0.0, GAME_LENGTH - scene_step)
    payoff_turns = max(1.0, turns_remaining - math.ceil(travel))

    if int(tgt[1]) == player:
        roi = 0.0
    else:
        roi = float(tgt[6]) * payoff_turns / (required + 1.0)

    enemy_pressure, ally_support = hb_local_pressure(tgt, planets, player)
    pressure_ratio = ally_support / (enemy_pressure + 0.10)

    # Defense urgency: target của mình nhưng pressure cao và ships thấp.
    if int(tgt[1]) == player:
        defense_urgency = clamp((enemy_pressure - ship_sigmoid(float(tgt[5]))) * 2.0, 0.0, 1.0)
    else:
        defense_urgency = 0.0

    # Counter risk: target có enemy mạnh gần đó, dễ bị retake.
    counter_risk = clamp(enemy_pressure / (ally_support + 0.25), 0.0, 2.0) / 2.0

    features = [
        clamp(travel / MAX_ETA),                                      # hb_intercept_eta_norm
        math.sin(angle),                                              # hb_intercept_angle_sin
        math.cos(angle),                                              # hb_intercept_angle_cos
        crosses_sun,                                                  # hb_crosses_sun
        first_hit_is_target,                                          # hb_first_hit_is_target
        min(required, 1000.0) / 1000.0,                               # hb_required_ships_norm
        clamp(required / (src_ships + EPS), 0.0, 2.0) / 2.0,          # hb_required_ratio
        clamp(surplus_after / 1000.0, -1.0, 1.0),                    # hb_surplus_after_send_norm
        source_safe,                                                  # hb_source_safe_after_send
        clamp(roi / 50.0),                                            # hb_roi_score
        clamp(payoff_turns / GAME_LENGTH),                            # hb_payoff_turns_norm
        defense_urgency,                                              # hb_defense_urgency
        clamp(enemy_pressure),                                        # hb_target_enemy_pressure
        clamp(ally_support),                                          # hb_target_ally_support
        clamp(pressure_ratio / 3.0),                                  # hb_local_pressure_ratio
        1.0 if hb_is_orbiting(tgt) else 0.0,                          # hb_target_is_orbiting
        source_frontline,                                             # hb_source_is_frontline
        counter_risk,                                                 # hb_counter_risk
    ]

    # Candidate hard-valid chỉ kiểm tra hình học thật sự nguy hiểm.
    # source_safe được giữ trong feature, không hard-mask, vì nhiều nước early-game
    # hợp lý cần rút xuống dưới ngưỡng garrison cũ.
    valid = (crosses_sun < 0.5) and (first_hit_is_target > 0.5)

    return {
        "valid": bool(valid),
        "angle": angle,
        "travel": travel,
        "required": required,
        "features": [0.0 if not math.isfinite(float(x)) else float(x) for x in features],
    }


def hb_candidate_valid(src, tgt, g):
    return hb_candidate_metrics(src, tgt, g)["valid"]


In [4]:
# =============================================================================
# BƯỚC 4 – REPLAY HISTORY (gọi một lần per step)
# =============================================================================
class ReplayHistory:
    """
    Theo dõi temporal features xuyên suốt một game.
    Quan trọng: phải gọi update() một lần cho mỗi step, trước vòng lặp candidate.
    """
    def __init__(self):
        self.ship_share     = {i: deque(maxlen=5) for i in range(4)}
        self.targeted       = deque(maxlen=5)
        self.incoming       = deque(maxlen=10)
        self.prev_enemy_dists = None

    def update(self, player, shares, targeted_ratio, hostile_incoming, enemy_dists):
        old_my   = self.ship_share.get(player, deque())
        momentum = shares.get(player, 0.0) - old_my[0] if len(old_my) >= 5 else 0.0

        enemy_momentums = []
        for o in range(4):
            if o == player:
                continue
            old = self.ship_share.get(o, deque())
            if len(old) >= 5:
                enemy_momentums.append(shares.get(o, 0.0) - old[0])

        fastest_grower_gap = max(enemy_momentums) - momentum if enemy_momentums else 0.0
        aggression_trend   = (targeted_ratio - self.targeted[0]
                              if len(self.targeted) >= 5 else 0.0)

        enemy_rhythm = 0.0
        if len(self.incoming) >= 10:
            vals = np.array(self.incoming, dtype=np.float32)
            enemy_rhythm = clamp(float(vals.var() / (vals.mean() ** 2 + EPS)))

        approach_rate = 0.0
        if self.prev_enemy_dists:
            vals = [max(0.0, self.prev_enemy_dists[o] - cur)
                    for o, cur in enemy_dists.items()
                    if o in self.prev_enemy_dists]
            if vals:
                approach_rate = clamp(sum(vals) / (len(vals) * 12.0 + EPS))

        for o in range(4):
            self.ship_share[o].append(shares.get(o, 0.0))
        self.targeted.append(targeted_ratio)
        self.incoming.append(hostile_incoming)
        self.prev_enemy_dists = dict(enemy_dists)

        return {
            "momentum":            clamp(momentum,           -1.0, 1.0),
            "fastest_grower_gap":  clamp(fastest_grower_gap, -1.0, 1.0),
            "aggression_trend":    clamp(aggression_trend,   -1.0, 1.0),
            "enemy_rhythm":        enemy_rhythm,
            "approach_rate":       approach_rate,
        }
    
# =============================================================================
# BƯỚC 5 – GLOBAL STATE PER STEP
# =============================================================================
def center_of_mass(planets, fleets, owner):
    """
    - Dùng cả planets lẫn fleets.
    Planet format: [id, owner, x, y, radius, ships, production]
    Fleet  format: [id, owner, x, y, angle, from_planet_id, ships]
    """
    sx = sy = sw = 0.0
    for p in planets:
        if int(p[1]) == owner:
            w   = max(1.0, float(p[5]))
            sx += p[2] * w;  sy += p[3] * w;  sw += w
    for f in fleets:
        if int(f[1]) == owner:
            w   = max(1.0, float(f[6]))
            sx += f[2] * w;  sy += f[3] * w;  sw += w
    return (sx / sw, sy / sw) if sw > 0 else (CENTER, CENTER)


def angular_spread(my_com, enemy_coms):
    if len(enemy_coms) <= 1:
        return 0.0
    angles = sorted(math.atan2(y - my_com[1], x - my_com[0]) for x, y in enemy_coms)
    gaps   = [angles[i+1] - angles[i] for i in range(len(angles)-1)]
    gaps.append(angles[0] + 2*math.pi - angles[-1])
    return clamp(1.0 - max(gaps) / (2*math.pi))


def compute_step_globals(step_idx, total_steps, player, planets, fleets, history):
    """
    Tính tất cả global state cho một step.
    Kết quả được pass vào extract_candidate_features cho mỗi candidate.
    """
    totals = defaultdict(lambda: {
        "planets": 0, "planet_ships": 0.0, "fleet_ships": 0.0,
        "prod": 0.0, "transit": 0.0, "total_ships": 0.0,
    })
    alive = set()

    for p in planets:
        owner = int(p[1])
        if owner == -1:
            continue
        totals[owner]["planets"]      += 1
        totals[owner]["planet_ships"] += float(p[5])
        totals[owner]["prod"]         += float(p[6])
        alive.add(owner)

    for f in fleets:
        owner = int(f[1])
        totals[owner]["fleet_ships"] += float(f[6])
        totals[owner]["transit"]     += float(f[6])
        alive.add(owner)

    for o in totals:
        totals[o]["total_ships"] = totals[o]["planet_ships"] + totals[o]["fleet_ships"]

    total_ships  = sum(v["total_ships"] for v in totals.values()) + EPS
    total_prod   = sum(v["prod"]        for v in totals.values()) + EPS
    my_total     = totals[player]["total_ships"]
    my_prod      = totals[player]["prod"]

    enemy_owners = [o for o in totals if o != player and totals[o]["total_ships"] > 0]
    max_enemy_ships = max((totals[o]["total_ships"] for o in enemy_owners), default=0.0)
    hostile_transit = sum(float(f[6]) for f in fleets if int(f[1]) != player)

    shares      = {o: totals[o]["total_ships"] / total_ships for o in range(4)}
    my_com      = center_of_mass(planets, fleets, player)
    enemy_coms  = {o: center_of_mass(planets, fleets, o) for o in enemy_owners}
    enemy_dists = {o: dist_xy(my_com[0], my_com[1], c[0], c[1])
                   for o, c in enemy_coms.items()}

    # Targeted ratio
    my_planet_ids  = {int(p[0]) for p in planets if int(p[1]) == player}
    hostile_to_me  = sum(float(f[6]) for f in fleets
                         if int(f[1]) != player
                         and int(f[5]) in my_planet_ids)
    targeted_ratio = clamp(hostile_to_me / (hostile_transit + EPS))

    hist = history.update(player, shares, targeted_ratio, hostile_transit, enemy_dists)

    weakest_enemy = (min(enemy_owners, key=lambda o: totals[o]["total_ships"])
                     if enemy_owners else None)

    avg_enemy_dist = (np.mean(list(enemy_dists.values())) / (math.sqrt(2) * BOARD)
                      if enemy_dists else 0.0)
    ang_spread = angular_spread(my_com, list(enemy_coms.values()))

    weakest_colony = 1.0
    my_planets = [p for p in planets if int(p[1]) == player]
    if my_planets:
        for p in my_planets:
            hostile_p = sum(float(f[6]) for f in fleets
                            if int(f[1]) not in (-1, player)
                            and int(f[5]) == int(p[0]))
            weakest_colony = min(weakest_colony,
                                 clamp((float(p[5]) - hostile_p) / (float(p[5]) + EPS)))
    else:
        weakest_colony = 0.0

    return {
        "totals":          totals,
        "total_ships":     total_ships,
        "total_prod":      total_prod,
        "my_total":        my_total,
        "my_prod":         my_prod,
        "max_enemy_ships": max_enemy_ships,
        "enemy_owners":    enemy_owners,
        "weakest_enemy":   weakest_enemy,
        "targeted_ratio":  targeted_ratio,
        "shares":          shares,
        "my_planets":      my_planets,
        "enemy_planets":   [p for p in planets if int(p[1]) not in (-1, player)],
        "num_planets":     len(planets),
        "avg_enemy_dist":  clamp(avg_enemy_dist),
        "angular_spread":  ang_spread,
        "weakest_colony":  weakest_colony,
        "hist":            hist,
        "game_progress":   clamp(step_idx / total_steps),
        "players_alive":   clamp(len(alive) / 4.0),
        "planets":         planets,
        "fleets":          fleets,
        "step_idx":        step_idx,
        "total_steps":     total_steps,
    }

In [5]:
# =============================================================================
# BƯỚC 6 – FEATURE EXTRACTION (46 cũ + 14 geometry + Hellburner features)
# =============================================================================

EXTRA_FEATURE_NAMES = [
    "src_x_norm",
    "src_y_norm",
    "tgt_x_norm",
    "tgt_y_norm",
    "dx_norm",
    "dy_norm",
    "dist_norm",
    "angle_sin",
    "angle_cos",
    "tgt_ships_norm",
    "src_ships_norm",
    "is_neutral",
    "is_self",
    "is_enemy",
]

# Chạy cell này sau khi đã khai báo FEATURE_NAMES 46 chiều cũ
FEATURE_NAMES = FEATURE_NAMES[:46] + EXTRA_FEATURE_NAMES + HELLBURNER_FEATURE_NAMES
FEATURE_DIM = len(FEATURE_NAMES)
print("FEATURE_DIM =", FEATURE_DIM)


def extract_candidate_features(src, tgt, raw_distance, g):
    """
    Trích feature cho cặp (src, tgt).
    - 46 feature đầu: chiến thuật/tổng quan cũ.
    - 14 feature tiếp theo: hình học + owner + ships.
    - phần cuối: Hellburner-style heuristics.
    """
    player    = int(src[1])
    src_id    = int(src[0])
    src_ships = max(1.0, float(src[5]))
    src_prod  = float(src[6])

    tgt_id    = int(tgt[0])
    tgt_owner = int(tgt[1])
    tgt_ships = max(0.0, float(tgt[5]))
    tgt_prod  = float(tgt[6])

    src_x = float(src[2])
    src_y = float(src[3])
    tgt_x = float(tgt[2])
    tgt_y = float(tgt[3])

    dx_raw = tgt_x - src_x
    dy_raw = tgt_y - src_y
    geom_dist = math.sqrt(dx_raw * dx_raw + dy_raw * dy_raw)
    angle_to_tgt = math.atan2(dy_raw, dx_raw)

    eta = max(1, int(math.ceil(raw_distance / SHIP_SPEED)))

    # Hellburner-style metrics cho candidate này.
    # Dùng lại trong phần feature cuối và cũng có thể dùng để mask candidate khi parse replay.
    hb = hb_candidate_metrics(src, tgt, g)
    hb_features = hb["features"]


    need   = 0.0 if tgt_owner == player else tgt_ships + 1.0
    margin = src_ships - need

    target_total = g["totals"][tgt_owner]["total_ships"] if tgt_owner != -1 else 0.0
    owner_power  = clamp(target_total / g["total_ships"])

    src_front = min(
        (dist_xy(src_x, src_y, e[2], e[3]) for e in g["enemy_planets"]),
        default=BOARD,
    )
    tgt_front = min(
        (dist_xy(tgt_x, tgt_y, e[2], e[3]) for e in g["enemy_planets"]),
        default=BOARD,
    )

    threat_potential = clamp(min(
        sum(
            ship_sigmoid(e[5]) * math.exp(
                -LAMBDA_POT * max(
                    1,
                    int(math.ceil(dist_xy(e[2], e[3], tgt_x, tgt_y) / SHIP_SPEED))
                )
            )
            for e in g["enemy_planets"]
        ),
        2.0
    ) / 2.0)

    support_potential = clamp(min(
        sum(
            ship_sigmoid(p[5]) * math.exp(
                -LAMBDA_POT * max(
                    1,
                    int(math.ceil(dist_xy(p[2], p[3], tgt_x, tgt_y) / SHIP_SPEED))
                )
            )
            for p in g["my_planets"]
            if int(p[0]) != src_id
        ),
        2.0
    ) / 2.0)

    if tgt_owner == player:
        diplomacy = 1.0
        survivors = float(src_ships)
    elif tgt_owner == -1:
        diplomacy = 0.0
        survivors = max(0.0, float(src_ships) - tgt_ships)
    else:
        diplomacy = -1.0
        survivors = max(0.0, float(src_ships) - tgt_ships)

    garrison_strength   = ship_sigmoid(survivors)
    defense_sust        = clamp(garrison_strength / (threat_potential + EPS))
    local_superiority   = clamp(min(support_potential / (threat_potential + 0.1), 3.0) / 3.0)

    projected_owner      = 1.0 if margin > 0 else (0.0 if tgt_owner == -1 else -1.0)
    projected_resistance = ship_sigmoid(max(tgt_ships - src_ships, 0.0))

    enemy_commitment = 0.0
    if tgt_owner not in (-1, player):
        t = g["totals"][tgt_owner]
        enemy_commitment = clamp(t["transit"] / (t["total_ships"] + EPS))

    enemy_exposed = clamp(enemy_commitment * (1.0 - owner_power))
    src_safety    = 1.0

    # ===== Các feature trước đây bị hard-code 0.0, giờ tính xấp xỉ thật =====
    # threat_after/support_after: pressure/support sau khi giả định source gửi required ships.
    hb_required = 0.0
    hb_travel = eta
    if isinstance(hb, dict):
        hb_required = float(hb.get("required", 0.0) or 0.0)
        hb_travel = float(hb.get("travel", eta) or eta)
        if not math.isfinite(hb_travel):
            hb_travel = eta

    remaining_src_ships = max(0.0, src_ships - hb_required)
    threat_after = clamp(threat_potential * (1.0 + 0.15 * (1.0 if tgt_owner not in (-1, player) else 0.0)))
    support_after = clamp(support_potential + ship_sigmoid(max(0.0, remaining_src_ships)) * math.exp(-max(1.0, hb_travel) / 35.0))
    pre_volatility = clamp((threat_potential + support_potential + abs(margin) / (src_ships + K_SHIPS)) / 3.0)

    # orbital_trend: target quay và đang đổi vị trí nhiều theo ETA thì cao hơn.
    cur_tx, cur_ty = tgt_x, tgt_y
    fut_tx, fut_ty = predict_rotated_xy(tgt, max(1, int(math.ceil(hb_travel))), g.get("angular_velocity", 0.0))
    orbital_trend = clamp(dist_xy(cur_tx, cur_ty, fut_tx, fut_ty) / 25.0)

    # convergence_threat: enemy gần target + target đang có giá trị cao.
    convergence_threat = clamp(threat_potential * (0.5 + 0.5 * clamp(tgt_prod / 5.0)))

    hist = g["hist"]

    row = [
        g["game_progress"],                                            # 0
        g["players_alive"],                                            # 1
        clamp(len(g["my_planets"]) / max(1, g["num_planets"])),        # 2
        clamp(g["my_prod"] / g["total_prod"]),                        # 3
        clamp(g["my_total"] / g["total_ships"]),                      # 4
        hist["momentum"],                                              # 5
        clamp(g["totals"][player]["transit"] / (g["my_total"] + EPS)), # 6
        clamp((g["max_enemy_ships"] - g["my_total"])
              / (g["my_total"] + K_SHIPS), -1.0, 2.0),                # 7
        g["targeted_ratio"],                                           # 8
        hist["fastest_grower_gap"],                                    # 9
        hist["aggression_trend"],                                      # 10
        hist["enemy_rhythm"],                                          # 11
        ship_sigmoid(src_ships),                                       # 12
        clamp(src_prod / 5.0),                                         # 13
        src_safety,                                                    # 14
        diplomacy,                                                     # 15
        clamp(tgt_prod / 5.0),                                         # 16
        1.0 if is_orbiting_planet(tgt) else 0.0,                       # 17
        owner_power,                                                   # 18
        1.0 if (tgt_owner == g["weakest_enemy"]
                and tgt_owner not in (-1, player)) else 0.0,           # 19
        projected_owner,                                               # 20
        projected_resistance,                                          # 21
        threat_after,                                                  # 22
        support_after,                                                 # 23
        pre_volatility,                                                # 24
        threat_potential,                                              # 25
        support_potential,                                             # 26
        clamp(eta / MAX_ETA),                                          # 27
        clamp(src_ships / (g["my_total"] + EPS)),                     # 28
        clamp(need / (src_ships + EPS), 0.0, 2.0),                     # 29
        clamp(margin / (src_ships + K_SHIPS), -1.0, 1.0),              # 30
        1.0 if src_ships > need else 0.0,                              # 31
        garrison_strength,                                             # 32
        defense_sust,                                                  # 33
        clamp(tgt_prod / (need + 1.0)),                                # 34
        clamp((src_front - tgt_front) / 60.0, -1.0, 1.0),              # 35
        g["avg_enemy_dist"],                                           # 36
        g["angular_spread"],                                           # 37
        orbital_trend,                                                 # 38
        convergence_threat,                                            # 39
        hist["approach_rate"],                                         # 40
        enemy_commitment,                                              # 41
        g["weakest_colony"],                                           # 42
        local_superiority,                                             # 43
        clamp(tgt_prod / (g["my_prod"] + 1.0)),                       # 44
        enemy_exposed,                                                 # 45

        # ===== 14 feature mới =====
        src_x / BOARD,                                                 # 46
        src_y / BOARD,                                                 # 47
        tgt_x / BOARD,                                                 # 48
        tgt_y / BOARD,                                                 # 49
        dx_raw / BOARD,                                                # 50
        dy_raw / BOARD,                                                # 51
        geom_dist / BOARD,                                             # 52
        math.sin(angle_to_tgt),                                        # 53
        math.cos(angle_to_tgt),                                        # 54
        min(tgt_ships, 1000.0) / 1000.0,                               # 55
        min(src_ships, 1000.0) / 1000.0,                               # 56
        1.0 if tgt_owner == -1 else 0.0,                               # 57
        1.0 if tgt_owner == player else 0.0,                           # 58
        1.0 if tgt_owner not in (-1, player) else 0.0,                 # 59

        # ===== Hellburner-style heuristic features =====
        *hb_features,
    ]

    return [0.0 if not math.isfinite(float(x)) else float(x) for x in row]

FEATURE_DIM = 78


In [6]:

# =============================================================================
# KIỂM TRA NHANH FEATURE DIM
# =============================================================================
print("FEATURE_DIM =", FEATURE_DIM)
print("Num feature names =", len(FEATURE_NAMES))
print("Hellburner features:", len(HELLBURNER_FEATURE_NAMES))
for i, name in enumerate(FEATURE_NAMES):
    print(f"{i:02d}: {name}")


FEATURE_DIM = 78
Num feature names = 78
Hellburner features: 18
00: game_progress
01: players_alive
02: my_planet_share
03: my_gdp_share
04: my_ship_share
05: momentum
06: my_fleet_ratio
07: leader_gap
08: am_i_targeted
09: fastest_grower_gap
10: aggression_trend
11: enemy_rhythm
12: src_ships
13: src_production
14: src_safety
15: target_diplomacy
16: target_production
17: is_comet
18: owner_power
19: is_weakest_target
20: projected_owner
21: projected_resistance
22: threat_after
23: support_after
24: pre_volatility
25: threat_potential
26: support_potential
27: eta
28: commitment_ratio
29: required_ratio
30: margin
31: can_win
32: garrison_strength
33: defense_sustainability
34: target_roi
35: front_gradient
36: avg_enemy_distance
37: angular_spread
38: orbital_trend
39: convergence_threat
40: approach_rate
41: enemy_commitment
42: weakest_colony
43: local_superiority
44: economic_impact
45: enemy_exposed
46: src_x_norm
47: src_y_norm
48: tgt_x_norm
49: tgt_y_norm
50: dx_norm
51: dy_n

In [7]:
import gc
from tqdm.auto import tqdm

MAX_LABEL_ANGLE_ERR = 0.05  # rad ~ 2.86 độ, lọc nhãn suy luận quá mơ hồ

def get_feature_dim():
    return len(FEATURE_NAMES)


def save_shard(X, MASK, Y, save_dir, shard_id, K, mode_name):
    os.makedirs(save_dir, exist_ok=True)

    out_path = os.path.join(save_dir, f"winner_each_replay_shard_{shard_id:04d}.npz")

    np.savez_compressed(
        out_path,
        X=np.asarray(X, dtype=np.float32),
        MASK=np.asarray(MASK, dtype=np.float32),
        Y=np.asarray(Y, dtype=np.int64),
        K=np.array(K),
        target_expert=np.array(mode_name),
        feature_names=np.array(FEATURE_NAMES),
    )

    print(f"Saved shard {shard_id}: {out_path} | samples={len(Y)}")


def choose_winner_from_rewards(replay):
    rewards = replay.get("rewards", None)

    if not rewards:
        return None

    clean_rewards = [
        float("-inf") if r is None else float(r)
        for r in rewards
    ]

    best = max(clean_rewards)
    if best == float("-inf"):
        return None

    return int(np.argmax(clean_rewards))


def find_player_snapshot(step, player_id):
    for player_state in step:
        obs = player_state.get("observation", {})
        if obs.get("player") == player_id:
            return player_state
    return None


def process_replays_sharded(
    data_folder,
    target_expert=None,
    K=24,
    save_dir="parsed_replay",
    shard_size=50000,
):
    X, MASK, Y = [], [], []
    json_files = sorted(glob.glob(os.path.join(data_folder, "*.json")))
    print(f"Tìm thấy {len(json_files)} replay.")

    total_actions = 0
    matched_actions = 0
    kept_samples = 0
    label_not_in_candidate = 0
    skipped_bad_angle = 0
    skipped_bad_src = 0
    skipped_unsafe_label = 0
    angle_errors = []

    shard_id = 0
    mode_name = "winner_each_replay" if target_expert is None else target_expert
    feature_dim = get_feature_dim()

    print(f"FEATURE_DIM = {feature_dim}")

    for file_path in tqdm(json_files, desc="Parsing replays"):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                replay = json.load(f)
        except MemoryError:
            print(f"MemoryError khi đọc file quá lớn, bỏ qua: {file_path}")
            gc.collect()
            continue
        except Exception as e:
            print(f"Lỗi đọc file {file_path}: {e}")
            continue

        team_names = replay.get("info", {}).get("TeamNames", [])

        if target_expert is not None:
            if target_expert not in team_names:
                del replay
                gc.collect()
                continue
            expert_id = team_names.index(target_expert)
        else:
            expert_id = choose_winner_from_rewards(replay)
            if expert_id is None:
                del replay
                gc.collect()
                continue

        total_steps = replay.get("configuration", {}).get("episodeSteps", 500)
        steps = replay.get("steps", [])
        history = ReplayHistory()

        # Quan trọng:
        # action ở steps[t] phải học từ observation ở steps[t-1]
        for step_idx in range(1, len(steps)):
            prev_snapshot = find_player_snapshot(steps[step_idx - 1], expert_id)
            cur_snapshot = find_player_snapshot(steps[step_idx], expert_id)

            if prev_snapshot is None or cur_snapshot is None:
                continue

            actions = cur_snapshot.get("action", [])
            if not actions:
                continue

            # Feature lấy từ state TRƯỚC khi action xảy ra
            obs = prev_snapshot.get("observation", {})
            planets = obs.get("planets", [])
            fleets = obs.get("fleets", [])
            angular_velocity = float(obs.get("angular_velocity", 0.0))

            if not planets:
                continue

            planet_by_id = {int(p[0]): p for p in planets}

            g = compute_step_globals(
                step_idx - 1,
                total_steps,
                expert_id,
                planets,
                fleets,
                history,
            )
            g["angular_velocity"] = angular_velocity

            for act in actions:
                total_actions += 1

                if len(act) < 3:
                    continue

                src_id = int(act[0])
                angle = float(act[1])
                ships = float(act[2])

                src = planet_by_id.get(src_id)

                # Vì obs là state trước action, source phải còn thuộc expert_id
                if src is None or int(src[1]) != expert_id:
                    skipped_bad_src += 1
                    continue

                true_target_id, err = infer_target_by_angle_lead(
                    src,
                    planets,
                    angle,
                    ships,
                    angular_velocity,
                )

                if true_target_id is None:
                    continue

                angle_errors.append(err)

                # Lọc các label suy từ angle quá mơ hồ
                if err > MAX_LABEL_ANGLE_ERR:
                    skipped_bad_angle += 1
                    continue

                matched_actions += 1

                candidates = build_candidates(src, planets, K=K)
                candidate_ids = [int(c[0][0]) for c in candidates[:K]]

                if true_target_id not in candidate_ids:
                    label_not_in_candidate += 1
                    continue

                features = np.zeros((K, feature_dim), dtype=np.float32)
                mask = np.zeros(K, dtype=np.float32)
                label = -1

                for i, (tgt, d) in enumerate(candidates[:K]):
                    feat = np.asarray(
                        extract_candidate_features(src, tgt, d, g),
                        dtype=np.float32,
                    )

                    if feat.shape[0] != feature_dim:
                        raise ValueError(
                            f"Feature dim mismatch: got {feat.shape[0]}, "
                            f"expected {feature_dim}. "
                            f"Kiểm tra FEATURE_NAMES và extract_candidate_features()."
                        )

                    features[i] = feat

                    # Hellburner guardrail:
                    # Chỉ hard-mask lỗi hình học chắc chắn sai: xuyên Sun / bắn nhầm planet.
                    # Source safety KHÔNG còn là hard-mask; nó chỉ là feature/penalty.
                    is_valid_by_hb = hb_candidate_valid(src, tgt, g)
                    mask[i] = 1.0 if is_valid_by_hb else 0.0

                    if int(tgt[0]) == true_target_id:
                        label = i

                if label >= 0:
                    # Nếu chính label expert bị hard guardrail đánh dấu unsafe,
                    # bỏ sample để tránh dạy model bắn xuyên Sun / bắn nhầm planet.
                    # Không bỏ chỉ vì source_safe thấp.
                    if mask[label] < 0.5:
                        skipped_unsafe_label += 1
                        continue

                    # Đảm bảo còn ít nhất một candidate hợp lệ.
                    if mask.sum() < 1:
                        skipped_unsafe_label += 1
                        continue

                    X.append(features)
                    MASK.append(mask)
                    Y.append(label)
                    kept_samples += 1

                if len(Y) >= shard_size:
                    save_shard(X, MASK, Y, save_dir, shard_id, K, mode_name)
                    shard_id += 1
                    X, MASK, Y = [], [], []
                    gc.collect()

        del replay
        gc.collect()

    if len(Y) > 0:
        save_shard(X, MASK, Y, save_dir, shard_id, K, mode_name)

    print("\nDone parsing.")
    print(f"total_actions:           {total_actions}")
    print(f"matched_actions:         {matched_actions} ({100 * matched_actions / (total_actions + 1):.1f}%)")
    print(f"kept_samples:            {kept_samples}")
    print(f"skipped_bad_src:         {skipped_bad_src}")
    print(f"skipped_bad_angle:       {skipped_bad_angle}")
    print(f"label_not_in_candidate:  {label_not_in_candidate}")
    print(f"skipped_unsafe_label:     {skipped_unsafe_label}")

    if len(angle_errors) > 0:
        arr = np.asarray(angle_errors, dtype=np.float32)
        print("\nAngle error stats:")
        print(f"mean: {arr.mean():.4f}")
        print(f"p50 : {np.percentile(arr, 50):.4f}")
        print(f"p90 : {np.percentile(arr, 90):.4f}")
        print(f"p95 : {np.percentile(arr, 95):.4f}")

In [8]:
# =============================================================================
# BƯỚC 8 – DATASET + TRANSFORMER
# =============================================================================
class CandidateDataset(Dataset):
    def __init__(self, X, MASK, Y):
        self.X    = torch.tensor(X,    dtype=torch.float32)
        self.MASK = torch.tensor(MASK, dtype=torch.bool)
        self.Y    = torch.tensor(Y,    dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.MASK[idx], self.Y[idx]
    

class CandidateTransformer(nn.Module):
    def __init__(self, feat_dim=None, d_model=128, nhead=4,
                 n_layers=6, dim_ff=256, dropout=0.1):
        super().__init__()
        feat_dim = FEATURE_DIM if feat_dim is None else int(feat_dim)
        self.feat_dim = feat_dim
        self.input_proj = nn.Linear(feat_dim, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.score_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x, mask):
        mask = mask.bool()
        x = self.input_proj(x)
        pad_mask = ~mask
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        scores = self.score_head(x).squeeze(-1)
        return scores.masked_fill(pad_mask, float("-inf"))

In [9]:
# =============================================================================
# BƯỚC 9 – TRAINING FROM MULTIPLE PARSED REPLAY SHARDS
# =============================================================================
from tqdm.auto import tqdm
import os
import glob
import gc
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


def load_shard(path):
    data = np.load(path, allow_pickle=True)
    X = data["X"].astype(np.float32)
    MASK = data["MASK"].astype(np.float32)
    Y = data["Y"].astype(np.int64)
    return X, MASK, Y


def compute_feature_stats_from_shards(shard_paths, max_shards=None):
    sum_x = None
    sum_x2 = None
    count = 0

    paths = shard_paths if max_shards is None else shard_paths[:max_shards]

    for path in tqdm(paths, desc="Computing mean/std"):
        X, MASK, Y = load_shard(path)

        flat_x = X.reshape(-1, X.shape[-1])
        flat_m = MASK.reshape(-1) > 0.5
        valid_x = flat_x[flat_m]

        if sum_x is None:
            sum_x = valid_x.sum(axis=0, dtype=np.float64)
            sum_x2 = (valid_x.astype(np.float64) ** 2).sum(axis=0)
        else:
            sum_x += valid_x.sum(axis=0, dtype=np.float64)
            sum_x2 += (valid_x.astype(np.float64) ** 2).sum(axis=0)

        count += valid_x.shape[0]

        del X, MASK, Y, flat_x, flat_m, valid_x
        gc.collect()

    mean = (sum_x / count).astype(np.float32)
    var = (sum_x2 / count) - (mean.astype(np.float64) ** 2)
    std = (np.sqrt(np.maximum(var, 1e-12)) + 1e-6).astype(np.float32)

    return mean, std


def normalize_features(X, mean, std):
    return ((X - mean[None, None, :]) / std[None, None, :]).astype(np.float32)


@torch.no_grad()
def evaluate_on_shards(model, val_shard_paths, criterion, device, mean, std, batch_size):
    model.eval()

    total_loss = 0.0
    total = 0
    top_1 = 0
    top_3 = 0

    for path in tqdm(val_shard_paths, desc="Validation", leave=False):
        X, MASK, Y = load_shard(path)
        X = normalize_features(X, mean, std)

        ds = CandidateDataset(X, MASK, Y)
        dl = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=(device.type == "cuda"),
        )

        for feat, mask, label in dl:
            feat = feat.to(device)
            mask = mask.to(device)
            label = label.to(device)

            scores = model(feat, mask)
            loss = criterion(scores, label)

            bs = label.size(0)
            total_loss += loss.item() * bs
            total += bs

            pred_1 = scores.argmax(dim=-1)
            top_1 += (pred_1 == label).sum().item()

            k = min(3, scores.size(1))
            pred_3 = scores.topk(k=k, dim=-1).indices
            top_3 += (pred_3 == label.unsqueeze(1)).any(dim=1).sum().item()

        del X, MASK, Y, ds, dl
        gc.collect()

    return {
        "loss": total_loss / max(1, total),
        "top1": top_1 / max(1, total),
        "top3": top_3 / max(1, total),
    }


def train_model_sharded(
    dataset_dir="/kaggle/input/datasets/nkt218/parsed-data",
    shard_pattern="winner_each_replay_shard_*.npz",
    epochs=20,
    batch_size=128,
    lr=3e-4,
    weight_decay=1e-4,
    device=None,
    save_dir="/kaggle/working/orbit_transformer",
    save_name="winner_each_replay_transformer_best.pt",
    val_ratio=0.10,
    seed=42,
    stats_max_shards=None,
):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, save_name)
    stats_path = os.path.join(save_dir, "feature_stats.npz")

    shard_paths = sorted(glob.glob(os.path.join(dataset_dir, shard_pattern)))
    if len(shard_paths) == 0:
        raise FileNotFoundError(f"Không tìm thấy shard nào ở: {dataset_dir}/{shard_pattern}")

    print("Device:", device)
    print("Dataset dir:", dataset_dir)
    print("Num shards:", len(shard_paths))

    X0, M0, Y0 = load_shard(shard_paths[0])
    K = X0.shape[1]
    feat_dim = X0.shape[2]
    print("Example shard:", shard_paths[0])
    print("Example X:", X0.shape, "MASK:", M0.shape, "Y:", Y0.shape)
    del X0, M0, Y0
    gc.collect()

    rng = np.random.default_rng(seed)
    shard_paths = np.array(shard_paths)
    rng.shuffle(shard_paths)

    val_size = max(1, int(len(shard_paths) * val_ratio))
    val_shards = list(shard_paths[:val_size])
    train_shards = list(shard_paths[val_size:])

    print("Train shards:", len(train_shards))
    print("Val shards:", len(val_shards))

    mean, std = compute_feature_stats_from_shards(train_shards, max_shards=stats_max_shards)

    np.savez_compressed(
        stats_path,
        mean=mean,
        std=std,
        K=np.array(K),
        feat_dim=np.array(feat_dim),
        feature_names=np.array(FEATURE_NAMES),
    )
    print("Saved feature stats:", stats_path)

    model = CandidateTransformer(feat_dim=feat_dim).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    criterion = nn.CrossEntropyLoss()

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, epochs),
        eta_min=lr * 0.05,
    )

    best_val_top1 = -1.0
    best_val_top3 = -1.0
    best_val_loss_for_best_top1 = float("inf")
    
    history = []
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        rng.shuffle(train_shards)

        total_loss = 0.0
        total = 0
        correct = 0

        epoch_bar = tqdm(train_shards, desc=f"Epoch {epoch}/{epochs}")

        for shard_path in epoch_bar:
            X, MASK, Y = load_shard(shard_path)
            X = normalize_features(X, mean, std)

            ds = CandidateDataset(X, MASK, Y)
            dl = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=(device.type == "cuda"),
            )

            for feat, mask, label in dl:
                feat = feat.to(device)
                mask = mask.to(device)
                label = label.to(device)

                scores = model(feat, mask)
                loss = criterion(scores, label)

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                bs = label.size(0)
                total_loss += loss.item() * bs
                total += bs

                pred = scores.argmax(dim=-1)
                correct += (pred == label).sum().item()

            train_loss_now = total_loss / max(1, total)
            train_acc_now = correct / max(1, total)

            epoch_bar.set_postfix({
                "loss": f"{train_loss_now:.4f}",
                "acc": f"{100 * train_acc_now:.1f}%",
            })

            del X, MASK, Y, ds, dl
            gc.collect()

        train_loss = total_loss / max(1, total)
        train_acc = correct / max(1, total)

        val_metrics = evaluate_on_shards(
            model=model,
            val_shard_paths=val_shards,
            criterion=criterion,
            device=device,
            mean=mean,
            std=std,
            batch_size=batch_size,
        )

        info = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top3": val_metrics["top3"],
        }
        history.append(info)

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"lr={current_lr:.2e} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={100 * train_acc:.2f}% | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_top1={100 * val_metrics['top1']:.2f}% | "
            f"val_top3={100 * val_metrics['top3']:.2f}%"
        )

        scheduler.step()

        val_loss = float(val_metrics["loss"])
        val_top1 = float(val_metrics["top1"])
        val_top3 = float(val_metrics["top3"])
        
        # Lưu theo val_top1.
        # Tie-break:
        #   1. val_top1 cao hơn
        #   2. nếu top1 bằng nhau gần như tuyệt đối, chọn val_top3 cao hơn
        #   3. nếu top3 cũng bằng, chọn val_loss thấp hơn
        is_better = (
            (val_top1 > best_val_top1)
            or (
                abs(val_top1 - best_val_top1) < 1e-12
                and val_top3 > best_val_top3
            )
            or (
                abs(val_top1 - best_val_top1) < 1e-12
                and abs(val_top3 - best_val_top3) < 1e-12
                and val_loss < best_val_loss_for_best_top1
            )
        )
        
        if is_better:
            best_val_top1 = val_top1
            best_val_top3 = val_top3
            best_val_loss_for_best_top1 = val_loss
        
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "feature_mean": mean,
                    "feature_std": std,
                    "K": K,
                    "feat_dim": feat_dim,
                    "history": history,
        
                    # Metric dùng để chọn checkpoint
                    "best_metric": "val_top1",
                    "best_val_top1": best_val_top1,
                    "best_val_top3": best_val_top3,
                    "best_val_loss": best_val_loss_for_best_top1,
        
                    # Training metadata
                    "label_smoothing": 0.0,
                    "lr_scheduler": "CosineAnnealingLR",
                    "train_shards": train_shards,
                    "val_shards": val_shards,
                },
                save_path,
            )
        
            print(
                f"Saved best checkpoint by val_top1: {save_path} | "
                f"top1={100 * best_val_top1:.2f}% | "
                f"top3={100 * best_val_top3:.2f}% | "
                f"loss={best_val_loss_for_best_top1:.4f}"
            )

    elapsed = time.time() - start

    print("\nDone.")
    print(f"Elapsed: {elapsed / 60:.2f} minutes")
    print(
        f"Best val_top1: {100 * best_val_top1:.2f}% | "
        f"val_top3: {100 * best_val_top3:.2f}% | "
        f"val_loss: {best_val_loss_for_best_top1:.4f}"
    )
    print(f"Checkpoint: {save_path}")

    return model, history

In [10]:

# =============================================================================
# BƯỚC 10 – TRAIN MODEL
# =============================================================================
# Nếu bạn vừa parse bằng cell trên, đổi dataset_dir thành:
# dataset_dir="/kaggle/working/parsed_hellburner_features"

model, history = train_model_sharded(
    dataset_dir="/kaggle/input/datasets/nkt218/parsed-data",
    shard_pattern="winner_each_replay_shard_*.npz",
    epochs=40,
    batch_size=128,
    lr=3e-4,
    device=None,
    save_dir="/kaggle/working/orbit_transformer_hb",
    save_name="winner_each_replay_transformer_hb_best.pt",
)


Device: cuda
Dataset dir: /kaggle/input/datasets/nkt218/parsed-data
Num shards: 6
Example shard: /kaggle/input/datasets/nkt218/parsed-data/winner_each_replay_shard_0000.npz
Example X: (50000, 24, 78) MASK: (50000, 24) Y: (50000,)
Train shards: 5
Val shards: 1


Computing mean/std:   0%|          | 0/5 [00:00<?, ?it/s]

Saved feature stats: /kaggle/working/orbit_transformer_hb/feature_stats.npz


/tmp/ipykernel_58/1853873655.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)


Epoch 1/40:   0%|          | 0/5 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 039/40 | lr=1.68e-05 | train_loss=0.7492 | train_acc=73.29% | val_loss=1.3321 | val_top1=61.92% | val_top3=85.55%


Epoch 40/40:   0%|          | 0/5 [00:00<?, ?it/s]

Validation:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 040/40 | lr=1.54e-05 | train_loss=0.7466 | train_acc=73.45% | val_loss=1.4153 | val_top1=61.47% | val_top3=85.08%

Done.
Elapsed: 32.60 minutes
Best val_top1: 63.25% | val_top3: 86.76% | val_loss: 1.1996
Checkpoint: /kaggle/working/orbit_transformer_hb/winner_each_replay_transformer_hb_best.pt


In [11]:
!zip -r orbit_trans_hb.zip /kaggle/working/orbit_transformer_hb

  adding: kaggle/working/orbit_transformer_hb/ (stored 0%)
  adding: kaggle/working/orbit_transformer_hb/winner_each_replay_transformer_hb_best.pt (deflated 8%)
  adding: kaggle/working/orbit_transformer_hb/feature_stats.npz (deflated 16%)


In [12]:
import json

metadata = {
    "title": "orbit-wars-transformer-hellburner-features",
    "id": "nkt218/orbit-wars-transformer-hellburner-features",
    "licenses": [{"name": "CC0-1.0"}]
}

with open("/kaggle/working/dataset-metadata.json", "w") as f:
    json.dump(metadata, f)


In [13]:
!kaggle datasets create -p /kaggle/working -r zip

Starting upload for file orbit_transformer_hb.zip
100%|██████████████████████████████████████| 2.91M/2.91M [00:00<00:00, 10.2MB/s]
Upload successful: orbit_transformer_hb.zip (3MB)
Starting upload for file orbit_trans_hb.zip
100%|██████████████████████████████████████| 2.91M/2.91M [00:00<00:00, 9.95MB/s]
Upload successful: orbit_trans_hb.zip (3MB)
Starting upload for file .virtual_documents.zip
100%|██████████████████████████████████████| 17.5k/17.5k [00:00<00:00, 66.2kB/s]
Upload successful: .virtual_documents.zip (18KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/nkt218/orbit-wars-transformer-hellburner-features
